# Sử dụng audio_validator.py để kiểm tra toàn bộ 2148 file và sinh invalid_audio.csv

In [1]:
import sys
from pathlib import Path

import pandas as pd
from tqdm import tqdm

project_root = Path.cwd().parent

sys.path.append(str(project_root))

from src.audio.audio_validator import check_audio

In [2]:
inventory_path = project_root / "data" / "metadata" / "data_inventory.csv"

inventory = pd.read_csv(inventory_path)

print(inventory.shape)
inventory.head()

(2148, 23)


,audio_id,audio_path,original_split,project_split,locale,speaker_id,normalized_speaker_id,speaker_sex,speaker_age,transcript,...,intent_idx,intent,scenario_str,duration_sec,sample_rate,num_channels,is_valid,protocol,role,checksum
0,9702,train-115/4dcf89cc7708ffe6339d97afd4da24f5.wav,train,UNUSED,vi-VN,657c8d982832af573ef2c039,NaN,Female,34,tôi muốn nghe một quyển sách bởi la quán trung,...,20,play_audiobook,play,4.68,48000,1,True,NaN,NaN,NaN
1,9671,train-115/2d48e259c29bdbf2039edfddad79cb61.wav,train,UNUSED,vi-VN,5cf03d69b094d700013e4d54,NaN,Female,31,bắt đầu phát tam quốc diễn nghĩa ở chỗ mà tôi ...,...,20,play_audiobook,play,4.56,48000,1,True,NaN,NaN,NaN
2,10249,train-115/81da41ae4fbf09485ad6a4214439f9d0.wav,train,UNUSED,vi-VN,5e25be7c5514e680ef436338,NaN,Female,33,hãy chơi một ván trivia,...,51,play_game,play,3.24,48000,1,True,NaN,NaN,NaN
3,3854,train-115/2866444482800feff8590246b56cd245.wav,train,UNUSED,vi-VN,657c8d982832af573ef2c039,NaN,Female,34,tắt loa làm ơn,...,46,audio_volume_mute,audio,2.58,48000,1,True,NaN,NaN,NaN
4,12053,train-115/ddd9fc1eee9336973c9de546c7af4794.wav,train,UNUSED,vi-VN,659ea35db097ea3c414b04c0,NaN,Female,31,các bộ phim được đánh giá cao đang chiếu cuối ...,...,55,recommendation_movies,recommendation,4.38,48000,1,True,NaN,NaN,NaN


In [3]:
audio_root = project_root / "data" / "audio"
print(audio_root)

d:\HCMUS\HOCTAP\Semesters\25-26HK3\HocThongKe\Project\VoiceStudy-Assistant\data\audio


In [4]:
example = audio_root / inventory.loc[0, "audio_path"]
print(example)
result = check_audio(str(example))
result

d:\HCMUS\HOCTAP\Semesters\25-26HK3\HocThongKe\Project\VoiceStudy-Assistant\data\audio\train-115\4dcf89cc7708ffe6339d97afd4da24f5.wav


{'exists': True,
 'readable': True,
 'duration': 4.68,
 'sample_rate': 48000,
 'channels': 1,
 'is_empty': False,
 'low_volume': False,
 'long_silence': False,
 'reason': 'valid'}

In [5]:
results = []

for _, row in tqdm(
    inventory.iterrows(),
    total=len(inventory),
    desc="Validating audio"
):

    audio_path = audio_root / row["audio_path"]
    result = check_audio(str(audio_path))
    result["audio_id"] = row["audio_id"]
    result["audio_path"] = row["audio_path"]
    results.append(result)

Validating audio: 100%|██████████| 2148/2148 [00:49<00:00, 43.67it/s]


### Tạo DataFrame

In [6]:
validation = pd.DataFrame(results)
print(validation.shape)
validation.head()

(2148, 11)


,exists,readable,duration,sample_rate,channels,is_empty,low_volume,long_silence,reason,audio_id,audio_path
0,True,True,4.68,48000,1,False,False,False,valid,9702,train-115/4dcf89cc7708ffe6339d97afd4da24f5.wav
1,True,True,4.56,48000,1,False,False,False,valid,9671,train-115/2d48e259c29bdbf2039edfddad79cb61.wav
2,True,True,3.24,48000,1,False,False,False,valid,10249,train-115/81da41ae4fbf09485ad6a4214439f9d0.wav
3,True,True,2.58,48000,1,False,False,False,valid,3854,train-115/2866444482800feff8590246b56cd245.wav
4,True,True,4.38,48000,1,False,True,False,low_volume,12053,train-115/ddd9fc1eee9336973c9de546c7af4794.wav


### Thống kê

In [11]:
summary = pd.DataFrame({
    "Metric": [
        "Total audio",
        "Valid audio",
        "Invalid audio"
    ],
    "Value": [
        len(validation),
        (validation["reason"] == "valid").sum(),
        (validation["reason"] != "valid").sum()
    ]
})

summary

,Metric,Value
0,Total audio,2148
1,Valid audio,2036
2,Invalid audio,112


In [7]:
validation["reason"].value_counts()

reason
valid           2036
low_volume       106
long_silence       5
empty_signal       1
Name: count, dtype: int64

In [8]:
invalid_audio = validation[
    validation["reason"] != "valid"
].copy()

print("Number of invalid files:", len(invalid_audio))
invalid_audio.head()

Number of invalid files: 112


,exists,readable,duration,sample_rate,channels,is_empty,low_volume,long_silence,reason,audio_id,audio_path
4,True,True,4.38,48000,1,False,True,False,low_volume,12053,train-115/ddd9fc1eee9336973c9de546c7af4794.wav
7,True,True,2.58,48000,1,False,True,False,low_volume,3948,train-115/36d3ce186ec4b08d21b1c0c0dfff44dd.wav
10,True,True,2.34,48000,1,False,True,False,low_volume,1100,train-115/b6183a1c39fe27478e903d5471695d9d.wav
14,True,True,2.94,48000,1,False,True,False,low_volume,12356,train-115/3bc7349db9598cda788b24325535b59e.wav
21,True,True,2.76,48000,1,False,True,False,low_volume,1155,train-115/1e3fb8cc6f1f9d92674f0ef67ba63aa9.wav


### Lưu invalid_audio.csv

In [9]:
output_dir = project_root / "data" / "metadata"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "invalid_audio.csv"

invalid_audio.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)
print(output_path)

d:\HCMUS\HOCTAP\Semesters\25-26HK3\HocThongKe\Project\VoiceStudy-Assistant\data\metadata\invalid_audio.csv


### Cập nhật data_inventory.csv

In [12]:
inventory["is_valid"] = validation["reason"] == "valid"

inventory.to_csv(
    inventory_path,
    index=False,
    encoding="utf-8-sig"
)

print("Updated:", inventory_path)

Updated: d:\HCMUS\HOCTAP\Semesters\25-26HK3\HocThongKe\Project\VoiceStudy-Assistant\data\metadata\data_inventory.csv
